In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType

#Define schema for sub-data

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("tier",StringType(), True)
])

survey_schema = StructType([
    StructField("question_id", StringType(), True),
    StructField("score", IntegerType(), True)
])

#Dataset wit plain and array structs
bpo_advanced_data = [
    ("INT-01", "Melany Rua", "Voice", ("CUST-99", "Premium"), [("Q1", 5), ("Q2", 4)]),
    ("INT-02", "Kevin Velez", "Chat", ("CUST-88", "Standard"), [("Q1", 3), ("Q2", 5), ("Q3", 4)]),
    ("INT-03", "Tamara Solorzano", "Voice", ("CUST-77", "Premium"), []) # Sin encuestas
]

#Main schema

main_schema = StructType([
    StructField("interaction_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("customer_details", customer_schema, True),
    StructField("survey", ArrayType(survey_schema), True)
])

#Create DataFrame
df_bpo = spark.createDataFrame(bpo_advanced_data, main_schema)
display(df_bpo)


In [0]:
# Extract customer data ddo flat format
df_customer_flat = df_bpo.select(
    "interaction_id",
    "customer_name",
    "channel", 
    F.col("customer_details.customer_id").alias("customer_id"),
    F.col("customer_details.tier").alias("tier")
)
display(df_customer_flat)

In [0]:
#Explode the survey column so each question have its own row

df_exploded_surveys = df_bpo.select(
    "channel",
    "interaction_id",
    F.explode_outer(F.col("survey")).alias("single_survey")   
)

df_analytics_ready = df_exploded_surveys.select(
    "interaction_id",
    F.col("single_survey.question_id").alias("question"),
    F.col("single_survey.score").alias("score")
)
display(df_analytics_ready)

In [0]:
#Now I will use the shema_of_json function to get the schema of the json column
df_bpo_json = df_bpo.withColumn("customer_json_string", F.to_json(F.col("customer_details")))\
    .withColumn("survey_json_string", F.to_json(F.col("survey")))\
    .drop("customer_details", "survey")

In [0]:
#First row to schema_of_json
first_row = df_bpo_json.first()
client_auto = first_row["customer_json_string"]
survey_auto = first_row["survey_json_string"]

client_schema = F.schema_of_json(F.lit(df_bpo_json.first()["customer_json_string"]))
survey_schema = F.schema_of_json(F.lit(df_bpo_json.first()["survey_json_string"]))

df_bpo_json = df_bpo_json.withColumn("client", F.from_json(F.col("customer_json_string"), client_schema)) \
                         .withColumn("survey", F.from_json(F.col("survey_json_string"), survey_schema)) \
                         .drop("customer_json_string", "survey_json_string")
display(df_bpo_json)
#Extract the customer details from the json column
df_bpo_json = df_bpo_json.select(
    "interaction_id",
    "customer_name",
    "channel",
    F.col("client.customer_id").alias("customer_id"),
    F.col("client.tier").alias("tier"),
    "survey"
)
display(df_bpo_json)
#Extract the survey details from the json column
df_bpo_json = df_bpo_json.select(
    "interaction_id",
    "customer_name",
    "channel",
    "customer_id",
    "tier",
    F.col("survey.question_id").alias("question"),
    F.col("survey.score").alias("score")
)
display(df_bpo_json)



In [0]:
#Explode df_bpo survey column
df_exploded = df_bpo.select(
    "interaction_id",
    "customer_name",
    F.explode_outer(F.col("survey")).alias("single_survey")
)

df_exploded = df_exploded.select(
    "interaction_id",
    "customer_name",
    F.col("single_survey.question_id").alias("question"),
    F.col("single_survey.score").alias("score")
)
display(df_exploded)